In [1]:
pwd

'C:\\Users\\hamza\\Programs\\Road_Traffic_Monitoring_AI_Vision_Smart_Sensor\\notebooks'

In [2]:
cd ..

C:\Users\hamza\Programs\Road_Traffic_Monitoring_AI_Vision_Smart_Sensor


C:\Users\hamza\anaconda3\envs\smart_sensor\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
import yaml
from geometry.primitives_pydantic import Line, SpeedLinePair, Polygon, Area, GeometryEngine
import numpy as np

def load_config(path: str):
    # =========================
    # 1. LOAD YAML
    # =========================
    with open(path, "r") as f:
        cfg = yaml.safe_load(f)

    # =========================
    # 2. BUILD LINES
    # =========================
    lines = {}
    for line_id, data in cfg.get("lines", {}).items():
        lines[line_id] = Line(
            line_id=line_id,
            points=tuple(map(tuple, data["points"])),
            vicinity=data.get("vicinity")
        )

    # =========================
    # 3. BUILD SPEED LINE PAIRS
    # =========================
    speed_pairs = {}
    for sp_id, data in cfg.get("speed_line_pairs", {}).items():
        speed_pairs[sp_id] = SpeedLinePair(
            line_pair_id=sp_id,
            line1=lines[data["line1_id"]],
            line2=lines[data["line2_id"]],
            distance_meters=data["distance_meters"]
        )

    # =========================
    # 4. BUILD POLYGONS
    # =========================
    polygons = {}
    areas = []

    for area_cfg in cfg.get("areas", []):
        geom = area_cfg.get("geometry", {})

        # --- Polygon (optional)
        poly_obj = None
        if "polygon" in geom:
            poly_data = geom["polygon"]
            poly_id = f"poly_{area_cfg['name']}"

            poly_obj = Polygon(
                polygon_id=poly_id,
                points=[tuple(p) for p in poly_data["points"]],
                distance_meters=poly_data.get("distance_meters")
            )
            polygons[poly_id] = poly_obj

        # --- Resolve references
        flow_line = lines.get(geom.get("flow_line_id"))
        speed_pair = speed_pairs.get(geom.get("speed_line_pair_id"))

        # --- Area object
        area = Area(
            name=area_cfg["name"],
            enable=area_cfg["enable"],
            description=area_cfg["description"],
            flow_line=flow_line,
            speed_pair=speed_pair,
            zone=poly_obj
        )

        areas.append(area)

    # =========================
    # 5. BUILD GEOMETRY ENGINE
    # =========================
    engine = GeometryEngine(
        lines=lines,
        polygons=polygons
    )

    # =========================
    # 6. RETURN STRUCTURED CONFIG
    # =========================
    return {
        "lines": lines,
        "speed_pairs": speed_pairs,
        "polygons": polygons,
        "areas": areas,
        "engine": engine,
        "metrics": cfg.get("metrics", {})
    }

In [4]:
cfg = load_config("configs/traffic_metrics.yaml")

cfg["engine"]      # GeometryEngine (used every frame)
cfg["areas"]       # List[Area] (resolved objects)
cfg["lines"]       # dict[str, Line]
cfg["speed_pairs"] # dict[str, SpeedLinePair]
cfg["metrics"]     # raw metrics config

{'general_params': {'in_polygon_mode': 'center_in_polygon',
  'homography_required': ['density', 'speed', 'headways_space']},
 'flow': {'enabled': True,
  'counting_logic': 'counter_5',
  'time_window_sec': 60,
  'output_unit': 'veh/h',
  'targets': [{'area': 'Direction_1'}, {'area': 'Direction_2'}]}}

In [5]:
engine = cfg["engine"]      # GeometryEngine (used every frame)


In [6]:
bboxes = np.array([
    [100, 100, 200, 200],
    [300, 300, 400, 400],
])

out = engine.compute(bboxes)

assert "line" in out
assert "polygon" in out

L = len(engine.line_ids)
N = len(bboxes)

assert out["line"]["distance"].shape == (L, N)
assert out["line"]["sign"].shape == (L, N)
assert out["line"]["vicinity_mask"].shape == (L, N)

for pid, mask in out["polygon"].items():
    assert mask.shape == (N,)

In [7]:
# fake line: y = 0
line = Line(line_id="L1", points=((0, 0), (10, 0)))

engine = GeometryEngine(
    lines={"L1": line},
    polygons={}
)

bboxes = np.array([
    [0, 10, 10, 20],   # center y > 0 → positive
    [0, -20, 10, -10], # center y < 0 → negative
])

out = engine.compute(bboxes)

signs = out["line"]["sign"][0]

assert signs[0] > 0
assert signs[1] < 0

In [8]:
out

{'line': {'distance': array([[ 15., -15.]]),
  'sign': array([[ 1., -1.]]),
  'vicinity_mask': array([[0, 0]])},
 'polygon': {}}

In [9]:
line = Line(
    line_id="L1",
    points=((0, 0), (10, 0)),
    vicinity=0.1
)

engine = GeometryEngine(lines={"L1": line}, polygons={})

bboxes = np.array([
    [0, 1, 10, 2],   # close to line
    [0, 100, 10, 110] # far
])

out = engine.compute(bboxes)

mask = out["line"]["vicinity_mask"][0]

assert mask[0] == 1
assert mask[1] == 0

In [10]:
poly = Polygon(
    polygon_id="P1",
    points=[(0,0), (100,0), (100,100), (0,100)]
)

engine = GeometryEngine(lines={}, polygons={"P1": poly})

bboxes = np.array([
    [10, 10, 20, 20],   # inside
    [200, 200, 300, 300] # outside
])

out = engine.compute(bboxes)

mask = out["polygon"]["P1"]

assert mask[0] == 1
assert mask[1] == 0

In [11]:
out

{'line': {'distance': array([], shape=(0, 2), dtype=float64),
  'sign': array([], shape=(0, 2), dtype=float64),
  'vicinity_mask': array([], shape=(0, 2), dtype=int64)},
 'polygon': {'P1': array([ True, False])}}

In [12]:
N = 10000
bboxes = np.random.randint(0, 1000, size=(N, 4))

out = engine.compute(bboxes)

# just ensure it runs fast and no shape issues
assert out["line"]["distance"].shape[1] == N

In [13]:
out

{'line': {'distance': array([], shape=(0, 10000), dtype=float64),
  'sign': array([], shape=(0, 10000), dtype=float64),
  'vicinity_mask': array([], shape=(0, 10000), dtype=int64)},
 'polygon': {'P1': array([False, False, False, ..., False, False, False], shape=(10000,))}}

In [14]:
track_ids = np.array([3, 4, 5, 7])
mask = np.array([True, True, False, True])
track_ids[mask]

array([3, 4, 7])